# 一、用 ERNIE-Image Turbo 生成 Cosplay 角色图

本教程参考 `ernie-image-tutorial/reference_proj.ipynb` 的组织方式，演示如何：

- 从角色数据集中按 `rank` 过滤出 **top-N**（默认 2000，可配置）
- **随机**挑一个角色的 `diffusion_prompt` 生成 cosplay 照片
- 按名字/作品名做**模糊搜索**，取第一个命中角色并生成
- 用 **Gradio** 做一个最小可用的交互 Demo

> 说明：本仓库里已有批量生成脚本（见 `scripts/ernie_cos/`），本 Notebook 更偏“教程 + 可交互演示”。


## 示例效果（仓库内已有生成结果）

下面 4 张图来自本仓库历史批跑输出（仅用于教程展示）。

- `ernie-image-tutorial/samples/304.jpg`
- `ernie-image-tutorial/samples/12393.jpg`
- `ernie-image-tutorial/samples/29282.jpg`
- `ernie-image-tutorial/samples/46465.jpg`


In [ ]:
from pathlib import Path

from PIL import Image

SAMPLES_DIR = Path("ernie-image-tutorial/samples")

sample_paths = [
    SAMPLES_DIR / "304.jpg",
    SAMPLES_DIR / "12393.jpg",
    SAMPLES_DIR / "29282.jpg",
    SAMPLES_DIR / "46465.jpg",
]

imgs = []
for p in sample_paths:
    if p.is_file():
        imgs.append(Image.open(p))
    else:
        print(f"missing: {p}")

imgs

# 二、环境与数据准备

## 1) 安装依赖

在 AIStudio / 本地环境中执行：

```bash
pip install --upgrade aistudio-sdk openai python-dotenv Pillow tqdm gradio
```

## 2) 下载数据集（AIStudio）

本教程使用数据集 `LC1332/anime-character-prompt-15k`，其中包含 `characters_top15000.jsonl`（每行一个角色，含 `rank` 与 `diffusion_prompt`）。

```bash
# 下载整个数据集到 ./data
aistudio download --dataset LC1332/anime-character-prompt-15k --local_dir ./data

# 或只下载单个文件（以 README.md 为例）
aistudio download README.md --dataset LC1332/anime-character-prompt-15k --local_dir ./data
```

## 3) 配置密钥

本仓库的脚本使用 `.env` 中的 `AISTUDIO_API_KEY`。
- AIStudio Notebook：建议在环境变量中配置 `AISTUDIO_API_KEY`
- 本地：在项目根目录创建 `.env`（不要提交到 git）


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# 本地：从项目根目录 .env 读取
# AIStudio：也可以直接在环境变量里配置 AISTUDIO_API_KEY
load_dotenv(Path(".env"))

AISTUDIO_API_KEY = (os.environ.get("AISTUDIO_API_KEY") or "").strip()
if not AISTUDIO_API_KEY:
    raise RuntimeError("缺少 AISTUDIO_API_KEY：请在环境变量或 .env 中设置")

client = OpenAI(
    api_key=AISTUDIO_API_KEY,
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",
)

"client ready"

# 三、数据集加载 + top-N 过滤（可配置）

我们从 `characters_top15000.jsonl` 读取每个角色的：

- `character_id`
- `name_cn` / `name_ja`
- `main_series`
- `rank`
- `diffusion_prompt`

然后按 `rank` 过滤出 top-N 作为候选池（默认 N=2000，可改）。


In [ ]:
import json
from dataclasses import dataclass


@dataclass(frozen=True)
class CharacterRow:
    character_id: int
    name_cn: str
    name_ja: str
    main_series: str
    rank: int
    diffusion_prompt: str


def load_characters_jsonl(jsonl_path: Path) -> list[CharacterRow]:
    out: list[CharacterRow] = []
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            out.append(
                CharacterRow(
                    character_id=int(obj["character_id"]),
                    name_cn=str(obj.get("name_cn") or ""),
                    name_ja=str(obj.get("name_ja") or ""),
                    main_series=str(obj.get("main_series") or ""),
                    rank=int(obj.get("rank") or 0),
                    diffusion_prompt=str(obj.get("diffusion_prompt") or ""),
                )
            )
    return out


# === 可配置参数 ===
TOP_N = 2000
IMAGE_SIZE = "768x1376"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 你的数据集下载位置：
# - AIStudio：通常在你指定的 local_dir 下
# - 本仓库：已有 local_data/characters_top15000.jsonl
JSONL_PATH_CANDIDATES = [
    Path("data/characters_top15000.jsonl"),
    Path("local_data/characters_top15000.jsonl"),
]

JSONL_PATH = next((p for p in JSONL_PATH_CANDIDATES if p.is_file()), None)
if JSONL_PATH is None:
    raise FileNotFoundError(
        "找不到 characters_top15000.jsonl，请确认已下载到 data/ 或 local_data/"
    )

rows = load_characters_jsonl(JSONL_PATH)
pool = [r for r in rows if 1 <= r.rank <= TOP_N]

len(rows), len(pool), pool[0]

# 四、核心函数：规范化 Prompt + 生成 JPG

这里把仓库脚本 `scripts/ernie_cos/common.py` 的核心逻辑做成“教程版简化实现”：

- 若 `diffusion_prompt` 的前两个关键词里都没有 `cosplay`，就把第一个关键词改为 `"{原第一个关键词} cosplay"`
- 调用 `ernie-image-turbo`，输出 `768x1376`，保存为 JPG
- 遇到偶发失败做少量重试


In [ ]:
import base64
import io
import random
import time
import urllib.request

from PIL import Image


def normalize_prompt(raw: str) -> str:
    s = (raw or "").strip()
    if not s:
        return ""

    parts = [p.strip() for p in s.split(",")]
    if not parts:
        return ""

    first = parts[0]
    second = parts[1] if len(parts) > 1 else ""

    def has_cosplay(x: str) -> bool:
        return "cosplay" in (x or "").casefold()

    if not has_cosplay(first) and not has_cosplay(second):
        parts[0] = f"{first} cosplay"

    return ", ".join(parts)


def _download_url(url: str) -> bytes:
    req = urllib.request.Request(
        url, headers={"User-Agent": "ernie-image-tutorial/1.0"}
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return resp.read()


def _bytes_to_jpeg(image_bytes: bytes, dest: Path) -> None:
    img = Image.open(io.BytesIO(image_bytes))
    rgb = img.convert("RGB")
    dest.parent.mkdir(parents=True, exist_ok=True)
    rgb.save(dest, format="JPEG", quality=95)


def generate_cos_jpg(
    *,
    character_id: int,
    prompt: str,
    dest_dir: Path,
    size: str = "768x1376",
    max_retries: int = 5,
    sleep_base: float = 2.0,
) -> Path:
    if not prompt.strip():
        raise ValueError("empty prompt")

    dest = dest_dir / f"{character_id}.jpg"

    last_err: Exception | None = None
    for attempt in range(max_retries):
        try:
            img = client.images.generate(
                model="ernie-image-turbo",
                prompt=prompt,
                n=1,
                response_format="b64_json",
                size=size,
                extra_body={
                    "seed": 42,
                    "use_pe": True,
                    "num_inference_steps": 8,
                    "guidance_scale": 1.0,
                },
            )
            item = img.data[0]

            b64 = getattr(item, "b64_json", None)
            if b64:
                raw = base64.b64decode(b64)
                _bytes_to_jpeg(raw, dest)
                return dest

            url = getattr(item, "url", None)
            if url:
                raw = _download_url(str(url))
                _bytes_to_jpeg(raw, dest)
                return dest

            raise RuntimeError("response has neither b64_json nor url")
        except Exception as e:
            last_err = e
            if attempt + 1 < max_retries:
                time.sleep(sleep_base * (2**attempt))

    raise RuntimeError(f"generate failed after retries: {last_err}")


"core helpers ready"

# 五、功能一：随机从 top-N 角色中挑一个生成

下面会从 `pool`（rank 1..TOP_N）随机挑一个角色：

- 显示角色信息
- 规范化 prompt（补 cosplay 关键词）
- 生成并保存到 `outputs/{character_id}.jpg`

> 注意：文生图存在“抽卡”，同一个角色多试几次可能差异较大。


In [ ]:
pick = random.choice(pool)

print("character_id:", pick.character_id)
print("rank:", pick.rank)
print("name_cn:", pick.name_cn)
print("name_ja:", pick.name_ja)
print("main_series:", pick.main_series)

prompt = normalize_prompt(pick.diffusion_prompt)
print("\nnormalized prompt (first 200 chars):\n", prompt[:200])

out_path = generate_cos_jpg(
    character_id=pick.character_id,
    prompt=prompt,
    dest_dir=OUTPUT_DIR,
    size=IMAGE_SIZE,
)

Image.open(out_path)

# 六、功能二：模糊搜索角色 → 生成第一个命中

目标：输入一个关键词（中文名 / 日文名 / 作品名均可），找到第一个“最像”的角色并生成。

实现策略（从强到弱）：

1) **近似匹配**：用 `difflib.get_close_matches` 在候选文本上找近似
2) **子串匹配**：若近似匹配为空，则在候选文本里做包含匹配

> 为了教程简洁，我们只取第一个命中。如果你希望“可选列表+手动选中”，可以在 Gradio 里扩展。


In [ ]:
import difflib


def _search_text(r: CharacterRow) -> str:
    parts = [r.name_cn, r.name_ja, r.main_series]
    return " ".join([p for p in parts if p]).strip()


def find_first_match(query: str, candidates: list[CharacterRow]) -> CharacterRow | None:
    q = (query or "").strip()
    if not q:
        return None

    texts = [_search_text(r) for r in candidates]

    # 1) 近似匹配（对短 query 更友好）
    close = difflib.get_close_matches(q, texts, n=5, cutoff=0.4)
    if close:
        target_text = close[0]
        idx = texts.index(target_text)
        return candidates[idx]

    # 2) 子串匹配（对作品名等更直观）
    q_fold = q.casefold()
    for r in candidates:
        if q_fold in _search_text(r).casefold():
            return r

    return None


# 试试："明日香" / "ひたぎ" / "命运石之门"
query = "明日香"
match = find_first_match(query, pool)
match

In [ ]:
if match is None:
    raise RuntimeError(f"没有找到匹配：{query!r}")

print("character_id:", match.character_id)
print("rank:", match.rank)
print("name_cn:", match.name_cn)
print("name_ja:", match.name_ja)
print("main_series:", match.main_series)

prompt = normalize_prompt(match.diffusion_prompt)
out_path = generate_cos_jpg(
    character_id=match.character_id,
    prompt=prompt,
    dest_dir=OUTPUT_DIR,
    size=IMAGE_SIZE,
)

Image.open(out_path)

# 七、Gradio Demo（随机 / 搜索）

这个 Demo 只做最小闭环：

- Tab 1：随机从 top-N 抽一个角色并生成
- Tab 2：输入关键词模糊搜索，取第一个命中并生成

生成结果会保存到 `outputs/`，并在界面中展示。

In [ ]:
import gradio as gr


def _format_info(r: CharacterRow) -> str:
    return (
        f"character_id: {r.character_id}\n"
        f"rank: {r.rank}\n"
        f"name_cn: {r.name_cn}\n"
        f"name_ja: {r.name_ja}\n"
        f"main_series: {r.main_series}\n"
    )


def gr_random_generate(top_n: int) -> tuple[str, Image.Image | None]:
    candidates = [r for r in rows if 1 <= r.rank <= int(top_n)]
    if not candidates:
        return f"top_n={top_n} 范围内没有角色，请调大 top_n", None

    r = random.choice(candidates)
    prompt = normalize_prompt(r.diffusion_prompt)
    out_path = generate_cos_jpg(
        character_id=r.character_id,
        prompt=prompt,
        dest_dir=OUTPUT_DIR,
        size=IMAGE_SIZE,
    )
    return _format_info(r), Image.open(out_path)


def gr_search_generate(query: str, top_n: int) -> tuple[str, Image.Image | None]:
    candidates = [r for r in rows if 1 <= r.rank <= int(top_n)]
    r = find_first_match(query, candidates)
    if r is None:
        return f"没有找到匹配：{query!r}", None

    prompt = normalize_prompt(r.diffusion_prompt)
    out_path = generate_cos_jpg(
        character_id=r.character_id,
        prompt=prompt,
        dest_dir=OUTPUT_DIR,
        size=IMAGE_SIZE,
    )
    return _format_info(r), Image.open(out_path)


with gr.Blocks(title="ERNIE-Image Cosplay 生成器（教程版）") as demo:
    gr.Markdown(
        """
        # ERNIE-Image Cosplay 生成器（教程版）

        - 默认从 top-N（按 rank）中抽取/搜索
        - 输出保存到 `outputs/`
        """
    )

    top_n = gr.Number(value=TOP_N, label="TOP_N（rank 上限）", precision=0)

    with gr.Tabs():
        with gr.TabItem("随机生成"):
            btn_rand = gr.Button("随机挑一个角色并生成", variant="primary")
            out_info_r = gr.Textbox(label="角色信息", lines=6)
            out_img_r = gr.Image(label="生成图")
            btn_rand.click(fn=gr_random_generate, inputs=[top_n], outputs=[out_info_r, out_img_r])

        with gr.TabItem("模糊搜索生成"):
            query = gr.Textbox(label="关键词（中文名 / 日文名 / 作品名）", placeholder="例如：明日香 / ひたぎ / 命运石之门")
            btn_search = gr.Button("搜索第一个命中并生成", variant="primary")
            out_info_s = gr.Textbox(label="角色信息", lines=6)
            out_img_s = gr.Image(label="生成图")
            btn_search.click(
                fn=gr_search_generate,
                inputs=[query, top_n],
                outputs=[out_info_s, out_img_s],
            )


demo

In [ ]:
# 在 AIStudio 建议用 share=True（按平台要求调整）
demo.launch(share=True)

# 八、批量生成（进阶）

如果你需要批量为一批角色生成图片（并跳过已生成的 id），建议直接使用仓库里的脚本，它们做了更完整的“去重 + 进度条 + 目录约定”：

- `scripts/ernie_cos/generate_top10.py`：先生成 top10 方便人工检查
- `scripts/ernie_cos/generate_rank_11_2600.py`：再批量生成 11-2600

核心公共逻辑在：`scripts/ernie_cos/common.py`

> 本 Notebook 的定位是：让你在 AIStudio 里快速跑通“随机/搜索→生成→保存→展示”的闭环，然后再切换到脚本做规模化批跑。
